# TechStack: Data Preparation and Water-Quality ML

This is the merged, reproducible handoff notebook for the complete data-to-model workflow. It cleans and validates the prepared data, trains three regression models for dissolved oxygen, compares them on a chronological holdout, saves evaluation figures in the root `images/` directory, and ends with the ML handoff.

**Scope:** no satellite features, no synthetic IoT ground truth, no DSS, CLI, or dashboard implementation.

## 1. Imports, paths, and reproducibility

The notebook can run from the repository root or from `notebooks/`.

In [ ]:
from pathlib import Path
import json
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
PROCESSED = ROOT / "data" / "processed"
MODELS = ROOT / "models"
IMAGES = ROOT / "images"
MODELS.mkdir(exist_ok=True)
IMAGES.mkdir(exist_ok=True)
print("Project root:", ROOT)
print("Output image directory:", IMAGES)

## 2. Load and inspect the prepared datasets

Weather remains a separate master dataset because historical date compatibility with water-quality observations has not been established.

In [ ]:
water_quality = pd.read_csv(PROCESSED / "water_quality_master.csv")
weather = pd.read_csv(PROCESSED / "weather_master.csv")
prepared_model = pd.read_csv(PROCESSED / "model_dataset.csv")
print("Water-quality master shape:", water_quality.shape)
print("Weather master shape:", weather.shape)
print("Existing model dataset shape:", prepared_model.shape)
print("Water-quality columns:", water_quality.columns.tolist())
print("Water-quality missing values:\n", water_quality.isna().sum())

## 3. Define the ML target and usable records

Dissolved oxygen is selected because it is the most complete continuous target in the prepared data. Rows with missing target or missing year are excluded from training because a chronological evaluation requires a known year. No missing values are imputed for the target, and the yearless source rows remain documented in the master dataset.

In [ ]:
TARGET = "dissolved_oxygen"
FEATURES = ["state", "station_name", "year"]
model_data = water_quality[FEATURES + [TARGET, "source_file"]].copy()
usable = model_data.dropna(subset=[TARGET, "year"]).copy()
usable["year"] = usable["year"].astype(int)
print("Target:", TARGET)
print("Usable records:", len(usable))
print("Excluded records:", len(model_data) - len(usable))
print("Usable years:", sorted(usable["year"].unique().tolist()))
print("Target summary:")
display(usable[TARGET].describe())

## 4. Target distribution and data checks

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(usable[TARGET], bins=20, color="#176b87", edgecolor="white")
plt.title("Dissolved Oxygen Target Distribution")
plt.xlabel("Dissolved oxygen")
plt.ylabel("Record count")
plt.tight_layout()
plt.savefig(IMAGES / "01_target_distribution.png", dpi=160)
plt.show()

print("Duplicate usable rows:", usable.duplicated().sum())
print("Target range:", float(usable[TARGET].min()), "to", float(usable[TARGET].max()))

## 5. Chronological train/test split

The latest observed year is held out for evaluation. This better reflects future prediction than a shuffled split. With the current data, 2019-2020 is the test period and earlier years are used for training.

In [ ]:
cutoff_year = usable["year"].quantile(0.80)
train = usable[usable["year"] < cutoff_year].copy()
test = usable[usable["year"] >= cutoff_year].copy()
if train.empty or test.empty:
    raise ValueError("Chronological split produced an empty train or test set.")
X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]
print("Train shape:", X_train.shape, "years:", sorted(train["year"].unique().tolist()))
print("Test shape:", X_test.shape, "years:", sorted(test["year"].unique().tolist()))

## 6. Preprocessing and model comparison

Every model uses the same fitted preprocessing pipeline. This prevents the earlier error where raw categorical state values were passed directly into a numeric estimator.

In [ ]:
categorical_features = ["state", "station_name"]
numerical_features = ["year"]
preprocessor = ColumnTransformer([
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), categorical_features),
    ("numeric", SimpleImputer(strategy="median"), numerical_features),
])
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1, min_samples_leaf=2),
    "Gradient Boosting": GradientBoostingRegressor(random_state=RANDOM_STATE, n_estimators=150, max_depth=2, learning_rate=0.04, loss="huber"),
}

fitted_pipelines = {}
predictions = {}
rows = []
for name, estimator in models.items():
    pipeline = Pipeline([("preprocessor", preprocessor), ("model", estimator)])
    pipeline.fit(X_train, y_train)
    pred = pipeline.predict(X_test)
    fitted_pipelines[name] = pipeline
    predictions[name] = pred
    rows.append({
        "model": name,
        "mae": mean_absolute_error(y_test, pred),
        "rmse": np.sqrt(mean_squared_error(y_test, pred)),
        "r2": r2_score(y_test, pred),
    })
results = pd.DataFrame(rows).sort_values(["rmse", "mae"]).reset_index(drop=True)
results.to_csv(PROCESSED / "model_comparison.csv", index=False)
display(results)

## 7. Compare models visually

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, metric in zip(axes, ["mae", "rmse", "r2"]):
    ordered = results.sort_values(metric, ascending=(metric != "r2"))
    ax.bar(ordered["model"], ordered[metric], color=["#176b87", "#e07a5f", "#3a7d44"][:len(ordered)])
    ax.set_title(metric.upper())
    ax.tick_params(axis="x", rotation=30)
    ax.grid(axis="y", alpha=0.25)
fig.suptitle("Chronological Holdout Model Comparison")
fig.tight_layout()
fig.savefig(IMAGES / "02_model_comparison.png", dpi=160)
plt.show()

best_model_name = results.iloc[0]["model"]
best_pipeline = fitted_pipelines[best_model_name]
print("Selected model by lowest RMSE, with MAE as tie-breaker:", best_model_name)

## 8. Best-model diagnostics

In [ ]:
best_predictions = predictions[best_model_name]
residuals = y_test.to_numpy() - best_predictions
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_test, best_predictions, alpha=0.75, color="#176b87")
lo = min(y_test.min(), best_predictions.min())
hi = max(y_test.max(), best_predictions.max())
ax.plot([lo, hi], [lo, hi], "--", color="#e07a5f", label="Perfect prediction")
ax.set_title(f"Actual vs Predicted Dissolved Oxygen: {best_model_name}")
ax.set_xlabel("Actual")
ax.set_ylabel("Predicted")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(IMAGES / "03_actual_vs_predicted.png", dpi=160)
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(best_predictions, residuals, alpha=0.75, color="#3a7d44")
ax.axhline(0, linestyle="--", color="#e07a5f")
ax.set_title(f"Residual Diagnostics: {best_model_name}")
ax.set_xlabel("Predicted dissolved oxygen")
ax.set_ylabel("Residual (actual - predicted)")
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(IMAGES / "04_residual_diagnostics.png", dpi=160)
plt.show()

## 9. Save model artifacts and traceable predictions

Only the selected best pipeline is marked as the primary artifact; all three fitted pipelines are saved for reproducibility and comparison.

In [ ]:
for name, pipeline in fitted_pipelines.items():
    safe_name = name.lower().replace(" ", "_")
    joblib.dump(pipeline, MODELS / f"{safe_name}_water_quality.joblib")

joblib.dump(best_pipeline, MODELS / "best_water_quality_model.joblib")
prediction_output = test[["state", "station_name", "year", TARGET, "source_file"]].copy()
prediction_output["prediction"] = best_predictions
prediction_output["residual"] = residuals
prediction_output.to_csv(PROCESSED / "best_model_predictions.csv", index=False)
metadata = {
    "target": TARGET,
    "features": FEATURES,
    "models_compared": list(models),
    "selected_model": best_model_name,
    "train_years": sorted(train["year"].unique().tolist()),
    "test_years": sorted(test["year"].unique().tolist()),
    "metrics": results.to_dict(orient="records"),
    "limitations": [
        "Weather was not merged because historical date compatibility is unestablished.",
        "Satellite features are excluded from the initial model.",
        "Synthetic IoT readings are not historical ground truth.",
    ],
}
(MODELS / "model_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Saved model artifacts, metrics, predictions, and metadata.")
print("Primary model:", MODELS / "best_water_quality_model.joblib")

## 10. Final ML handoff

The ML task ends here with trained and evaluated model artifacts. Future work begins with integration, DSS/risk logic, CLI, and system testing. Satellite remains excluded from the initial ML model.

In [ ]:
print("ML HANDOFF COMPLETE")
print("Target:", TARGET)
print("Selected model:", best_model_name)
print("Comparison:")
display(results)
print("Images:", sorted(p.name for p in IMAGES.glob("*.png")))
print("Models:", sorted(p.name for p in MODELS.glob("*.joblib")))